# FixedChunker Demo
Shows character-mode and token-mode chunking on a medium-length passage.

## Imports

In [1]:
# Standard Library
# Third Party Library
# Private Library
from cleave.chunker.fixed import FixedChunker
from cleave.schemas import (
    ChunkParams, ChunkUnit,
    ContentBlock, ContentType,
    Document, DocumentPage, Source, SourceType,
)

## Data

In [2]:
TEXT = """\
Chunking is the process of splitting a long document into smaller, overlapping pieces
so that each piece fits within the context window of a language model. The overlap
ensures that no information is lost at the boundary between two adjacent chunks —
a sentence or phrase that straddles a boundary will appear in both neighbours.

Fixed chunking is the simplest strategy: a sliding window of fixed size moves over
the text with a fixed step. The step is always smaller than the window, which creates
the overlap. The window can be measured in characters or in tokens depending on the
use case. Token-based chunking is more precise for LLM context limits; character-based
chunking is faster and requires no tokeniser.
"""

In [3]:
print(f"Text length : {len(TEXT)} chars")
print(f"Preview : {TEXT[:80]}...")

Text length : 717 chars
Preview : Chunking is the process of splitting a long document into smaller, overlapping p...


In [4]:
source = Source(type=SourceType.txt, name="demo.txt", location="/tmp/demo.txt")

In [5]:
def make_document(text: str) -> Document:
    page = DocumentPage(
        page_number=1,
        blocks=[ContentBlock(type=ContentType.text, content=text, position=0)],
    )
    return Document(source=source, pages=[page], total_pages=1)

## Character mode

In [6]:
CHUNK_SIZE = 120
CHUNK_OVERLAP = 20

In [7]:
char_chunker = FixedChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP))
char_chunks = char_chunker.chunk(make_document(TEXT))

In [8]:
print(f"chunk_size={CHUNK_SIZE}  chunk_overlap={CHUNK_OVERLAP}  step={CHUNK_SIZE - CHUNK_OVERLAP}")
print(f"Total chunks : {len(char_chunks)}\n")

for c in char_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}–{c.char_end:<3}  tokens={c.token_count:>3}  │ {c.text[:60]!r}")

chunk_size=120  chunk_overlap=20  step=100
Total chunks : 7

[0] chars   0–120  tokens= 23  │ 'Chunking is the process of splitting a long document into sm'
[1] chars 100–220  tokens= 24  │ 'iece fits within the context window of a language model. The'
[2] chars 200–320  tokens= 25  │ 'lost at the boundary between two adjacent chunks —\na sentenc'
[3] chars 300–420  tokens= 25  │ 'll appear in both neighbours.\n\nFixed chunking is the simples'
[4] chars 400–520  tokens= 28  │ 'ze moves over\nthe text with a fixed step. The step is always'
[5] chars 500–620  tokens= 26  │ 'the overlap. The window can be measured in characters or in '
[6] chars 600–717  tokens= 25  │ 'based chunking is more precise for LLM context limits; chara'


### Overlap inspection
The last `chunk_overlap` characters of chunk *n* should match the first `chunk_overlap` characters of chunk *n+1*.

In [9]:
for a, b in zip(char_chunks, char_chunks[1:]):
    tail = a.text[-CHUNK_OVERLAP:]
    head = b.text[:CHUNK_OVERLAP]
    match = "✓" if tail == head else "✗"
    print(f"chunk {a.index}→{b.index}  {match}  overlap: {tail!r}")

chunk 0→1  ✓  overlap: 'iece fits within the'
chunk 1→2  ✓  overlap: 'lost at the boundary'
chunk 2→3  ✓  overlap: 'll appear in both ne'
chunk 3→4  ✓  overlap: 'ze moves over\nthe te'
chunk 4→5  ✓  overlap: 'the overlap. The win'
chunk 5→6  ✓  overlap: 'based chunking is mo'


## Token mode

In [10]:
TOKEN_SIZE = 40
TOKEN_OVERLAP = 8

In [11]:
tok_chunker = FixedChunker(
    ChunkParams(chunk_size=TOKEN_SIZE, chunk_overlap=TOKEN_OVERLAP, unit=ChunkUnit.tokens)
)
tok_chunks = tok_chunker.chunk(make_document(TEXT))

In [12]:
print(f"chunk_size={TOKEN_SIZE} tokens  chunk_overlap={TOKEN_OVERLAP} tokens  step={TOKEN_SIZE - TOKEN_OVERLAP}")
print(f"Total chunks : {len(tok_chunks)}\n")

for c in tok_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}–{c.char_end:<3}  tokens={c.token_count:>3}  │ {c.text[:60]!r}")

chunk_size=40 tokens  chunk_overlap=8 tokens  step=32
Total chunks : 5

[0] chars   0–204  tokens= 40  │ 'Chunking is the process of splitting a long document into sm'
[1] chars 168–370  tokens= 40  │ '\nensures that no information is lost at the boundary between'
[2] chars 328–512  tokens= 40  │ '.\n\nFixed chunking is the simplest strategy: a sliding window'
[3] chars 477–670  tokens= 40  │ ' window, which creates\nthe overlap. The window can be measur'
[4] chars 630–717  tokens= 19  │ ' for LLM context limits; character-based\nchunking is faster '


## Character Unit Comparison

In [13]:
print(f"{'Mode':<12} {'Chunks':>6}  {'Avg chars':>10}  {'Avg tokens':>10}")
print("-" * 44)

for label, chunks in [("characters", char_chunks), ("tokens", tok_chunks)]:
    avg_chars = sum(len(c.text) for c in chunks) / len(chunks)
    avg_tokens = sum(c.token_count for c in chunks) / len(chunks)
    print(f"{label:<12} {len(chunks):>6}  {avg_chars:>10.1f}  {avg_tokens:>10.1f}")

Mode         Chunks   Avg chars  Avg tokens
--------------------------------------------
characters        7       119.6        25.1
tokens            5       174.0        35.8
